#### Rank sweep against full fine-tuning and loss comparison

In [3]:
!pip install import-ipynb
import import_ipynb

In [10]:
import lab06_starter as S
from loss_and_baseline import answer_only_labels
from lora_layer import replace_linear_with_lora
import torch

In [ ]:
def run_piece3():
    print("--- Piece 3: Rank Sweep & Loss Comparison ---")
    train_ex, test_ex, label_ids = S.load_task("emotion", n_train=600, n_test=500)
    seeds = [0, 1, 2]
    steps = 120
    batch_size = 8

    # Learning rates (tuned or standard per guidelines)
    # Full fine-tuning typically uses lower lr (e.g., 5e-5), LoRA uses higher lr (e.g., 1e-3)
    configs = [
        {"mode": "none", "name": "Zero-Shot", "lr": 0, "labels_fn": answer_only_labels},
        {"mode": "full", "name": "Full Fine-Tune", "lr": 5e-5, "labels_fn": answer_only_labels},
        {"mode": "lora", "r": 1, "name": "LoRA r=1", "lr": 1e-3, "labels_fn": answer_only_labels},
        {"mode": "lora", "r": 4, "name": "LoRA r=4", "lr": 1e-3, "labels_fn": answer_only_labels},
        {"mode": "lora", "r": 16, "name": "LoRA r=16", "lr": 1e-3, "labels_fn": answer_only_labels},
        {"mode": "lora", "r": 16, "name": "LoRA r=16 (All-Token)", "lr": 1e-3, "labels_fn": S.all_tokens_labels},
    ]

    results = []
    for cfg in configs:
        accuracies = []
        for seed in seeds:
            torch.manual_seed(seed)
            model, tok = S.load_model()
            
            if cfg["mode"] == "lora":
                replace_linear_with_lora(model, r=cfg["r"], alpha=2 * cfg["r"])
            elif cfg["mode"] == "none":
                info = dict(accuracy=S.evaluate(model, tok, test_ex, label_ids), trainable=0)
                accuracies.append(info["accuracy"])
                continue

            info = S.train(model, tok, train_ex, steps=steps, lr=cfg["lr"], batch_size=batch_size, 
                           seed=seed, labels_fn=cfg["labels_fn"])
            acc = S.evaluate(model, tok, test_ex, label_ids)
            accuracies.append(acc)
            
        mean_acc = sum(accuracies) / len(accuracies)
        print(f"{cfg['name']} | Accuracies: {[round(a, 3) for a in accuracies]} | Mean: {mean_acc:.3f}")
        results.append((cfg['name'], accuracies, mean_acc))

if __name__ == "__main__":
    run_piece3()

--- Piece 3: Rank Sweep & Loss Comparison ---


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Zero-Shot | Accuracies: [0.326, 0.326, 0.326] | Mean: 0.326


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

... (6 outputs left out)

LoRA r=1 | Accuracies: [0.716, 0.708, 0.722] | Mean: 0.715


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]